# Backfill 30 days of partitions without double-counting and without advancing the active bookmark

The pipeline has been quietly wrong for two weeks. The daily aggregation job threw a divide-by-zero on a cohort with zero spend, the cell crashed mid-run, the partition for that day landed empty, and the next 29 daily runs all ran cleanly on top of an already-empty target. Finance has been reading wrong numbers for 30 days. They noticed Tuesday.

You have one session to fix 30 days of partitions without touching the live schedule's bookmark and without writing duplicates if the backfill retries. The parity proof is the deliverable; the bookmark integrity is the constraint. Both are non-negotiable.

This is the production reality the cert version of "Glue job bookmarks" never asks you to handle. DEA-C01 Lab 07 taught you what a bookmark is. This lab is what you do when the bookmark is correct and the data behind it is wrong.

The five deliverables map to five checkpoints:
1. The live incremental job ran today against the seed bronze data; the bookmark advanced.
2. The backfill job is configured with bookmarks DISABLED and scoped to a 30-day window via job parameters.
3. The backfill execution (via Step Functions) rebuilds the 30-day window in `daily_revenue_agg`; the Glue job scanned under 30 GB of bronze.
4. The live incremental bookmark is UNCHANGED after the backfill (the interview talking point).
5. Row-level parity against `daily_revenue_truth` on the 30-day window returns zero mismatches.

**Time.** About 75 minutes hands-on. Two real Glue ETL job runs at the 10-minute minimum each is most of the wall clock.

**Cost.** About 80 cents on a clean run. Glue ETL is the line item: two 2-DPU runs at the 10-minute minimum is roughly 30 cents apiece. Parity Athena diff is pennies. Step Functions is free at lab volume. Realistic upper bound: $1.50.

**No critical resources.** Glue ETL DPU-minutes only bill while a job is running.

This lab is part of the Labrun Data Engineering job-prep track, Pipeline Engineering sub-track.


In [ ]:
!pip install --quiet labrun-checks==0.8.0 boto3


In [ ]:
# Imports and constants.

import atexit
import io
import json
import re
import sys
import time
import uuid
import zipfile
from datetime import datetime, timedelta
from getpass import getpass

import boto3
from botocore.exceptions import ClientError

from labrun_checks import CheckpointResult, CleanupResource, check, cleanup, register, run_cleanup

LAB_SLUG = "idempotent-backfill-30-days"
LAB_ID = LAB_SLUG
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_SLUG
REGION = "us-east-1"

BUCKET_NAME = None
DATABASE_NAME = f"labrun_{LAB_SLUG.replace('-', '_')}_db"
BRONZE_TABLE = "events_bronze"
AGG_TABLE = "daily_revenue_agg"
TRUTH_TABLE = "daily_revenue_truth"

INCREMENTAL_JOB = f"labrun-{LAB_SLUG}-incremental"
BACKFILL_JOB = f"labrun-{LAB_SLUG}-backfill"
ROLE_NAME = f"labrun-{LAB_SLUG}-role"
INLINE_POLICY_NAME = "labrun-glue-inline"
SFN_NAME = f"labrun-{LAB_SLUG}-sm"
SFN_ROLE = f"labrun-{LAB_SLUG}-sfn-role"
DAILY_RULE = f"labrun-{LAB_SLUG}-daily"
ATHENA_WG = f"labrun-{LAB_SLUG}-wg"

# Backfill window: the past 30 days.
BACKFILL_END = datetime(2026, 5, 14).date()
BACKFILL_START = BACKFILL_END - timedelta(days=29)

ACCOUNT_ID = None
INITIAL_BOOKMARK_RUN_ID = None
SM_ARN = None
SFN_ROLE_ARN = None


In [ ]:
# NBVAL_SKIP
# Setup.

print("Paste your lab session token:")
SESSION_TOKEN = getpass("Session token: ").strip()
if not SESSION_TOKEN:
    raise SystemExit("Missing session token.")
AWS_ACCESS_KEY_ID = getpass("AWS_ACCESS_KEY_ID: ").strip()
AWS_SECRET_ACCESS_KEY = getpass("AWS_SECRET_ACCESS_KEY: ").strip()
AWS_SESSION_TOKEN = getpass("AWS_SESSION_TOKEN (Enter to skip): ").strip() or None
AWS_CREDENTIALS = {"aws_access_key_id": AWS_ACCESS_KEY_ID, "aws_secret_access_key": AWS_SECRET_ACCESS_KEY, "aws_session_token": AWS_SESSION_TOKEN}

sts = boto3.client("sts", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})
try:
    ACCOUNT_ID = sts.get_caller_identity()["Account"]
except ClientError as exc:
    raise SystemExit(f"AWS credentials rejected: {exc}")

BUCKET_NAME = f"labrun-{LAB_SLUG}-{ACCOUNT_ID}"
print(f"Account: {ACCOUNT_ID}; bucket: {BUCKET_NAME}")
register(SESSION_TOKEN, lab_id=LAB_ID, credentials=AWS_CREDENTIALS)
print("Setup complete.")


In [ ]:
# NBVAL_SKIP
# CLEANUP_MANIFEST + atexit + orphan scan.

CLEANUP_MANIFEST = [
    CleanupResource(type="sfn_state_machine", id="pending", region=REGION),
    CleanupResource(type="eventbridge_rule", id=DAILY_RULE, region=REGION),
    CleanupResource(type="glue_job", id=INCREMENTAL_JOB, region=REGION,
        cli_delete_command=f"aws glue delete-job --job-name {INCREMENTAL_JOB} --region {REGION}"),
    CleanupResource(type="glue_job", id=BACKFILL_JOB, region=REGION,
        cli_delete_command=f"aws glue delete-job --job-name {BACKFILL_JOB} --region {REGION}"),
    CleanupResource(type="athena_workgroup", id=ATHENA_WG, region=REGION,
        cli_delete_command=f"aws athena delete-work-group --work-group {ATHENA_WG} --recursive-delete-option --region {REGION}"),
    CleanupResource(type="iceberg_table", id=f"{DATABASE_NAME}.{TRUTH_TABLE}", region=REGION),
    CleanupResource(type="iceberg_table", id=f"{DATABASE_NAME}.{AGG_TABLE}", region=REGION),
    CleanupResource(type="iceberg_table", id=f"{DATABASE_NAME}.{BRONZE_TABLE}", region=REGION),
    CleanupResource(type="iam_role", id=SFN_ROLE, region=REGION),
    CleanupResource(type="iam_role", id=ROLE_NAME, region=REGION),
    CleanupResource(type="glue_database", id=DATABASE_NAME, region=REGION),
    CleanupResource(type="s3_bucket", id=BUCKET_NAME, region=REGION),
]


def _atexit_cleanup():
    print("[atexit] Glue ETL bills only while running, but verify cleanup ran.")


atexit.register(_atexit_cleanup)

tagging = boto3.client("resourcegroupstaggingapi", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})
try:
    orphan_arns = [r["ResourceARN"] for r in tagging.get_resources(TagFilters=[{"Key": LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}]).get("ResourceTagMappingList", [])]
except ClientError as exc:
    raise SystemExit(f"Orphan scan failed: {exc}")

if orphan_arns:
    print("Found orphans from a previous session. Cleanup is blocking.")
    for arn in orphan_arns:
        print(f"  ORPHAN: {arn}")
    raise SystemExit("Run previous session's cleanup cell first.")

print("No orphans. Manifest declared.")


In [ ]:
# NBVAL_SKIP
# Seed bronze + truth. Bronze is wrong for the last 30 days (empty
# partitions per the scenario); the truth table has the correct values.

s3 = boto3.client("s3", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})
iam = boto3.client("iam", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})
glue = boto3.client("glue", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})
events = boto3.client("events", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})
athena = boto3.client("athena", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})
sfn = boto3.client("stepfunctions", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})

try:
    s3.create_bucket(Bucket=BUCKET_NAME)
    s3.put_bucket_tagging(Bucket=BUCKET_NAME, Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]})
except ClientError as exc:
    if exc.response["Error"]["Code"] != "BucketAlreadyOwnedByYou":
        raise

try:
    glue.create_database(DatabaseInput={"Name": DATABASE_NAME})
except glue.exceptions.AlreadyExistsException:
    pass

try:
    athena.create_work_group(
        Name=ATHENA_WG,
        Configuration={"ResultConfiguration": {"OutputLocation": f"s3://{BUCKET_NAME}/athena/"}, "EnforceWorkGroupConfiguration": True},
        Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
    )
except Exception:
    pass


def run_athena(query, timeout=60):
    qid = athena.start_query_execution(QueryString=query, WorkGroup=ATHENA_WG, QueryExecutionContext={"Database": DATABASE_NAME})["QueryExecutionId"]
    deadline = time.time() + timeout
    while time.time() < deadline:
        s = athena.get_query_execution(QueryExecutionId=qid)["QueryExecution"]["Status"]["State"]
        if s == "SUCCEEDED":
            return qid
        if s in ("FAILED", "CANCELLED"):
            raise RuntimeError(f"Athena {qid} {s}: {athena.get_query_execution(QueryExecutionId=qid)['QueryExecution']['Status'].get('StateChangeReason')}")
        time.sleep(2)
    raise TimeoutError()


# Seed bronze JSON to s3://BUCKET/raw/ for 60 days of events.
import random
rnd = random.Random(42)
all_lines = []
for d in range(60):
    day = (BACKFILL_END - timedelta(days=d))
    n = rnd.randint(50, 150)
    for i in range(n):
        all_lines.append(json.dumps({
            "event_id": f"{day.isoformat()}-{i}",
            "event_date": day.isoformat(),
            "customer_id": f"cust-{rnd.randint(1, 200)}",
            "revenue_cents": rnd.randint(100, 50000),
        }))
s3.put_object(Bucket=BUCKET_NAME, Key="raw/events.jsonl", Body="\n".join(all_lines).encode("utf-8"))
print(f"  bronze seed: {len(all_lines)} events across 60 days")

# Bronze Iceberg table
run_athena(
    f"CREATE TABLE IF NOT EXISTS {DATABASE_NAME}.{BRONZE_TABLE} ("
    f"  event_id string, event_date date, customer_id string, revenue_cents bigint"
    f") LOCATION 's3://{BUCKET_NAME}/{BRONZE_TABLE}/' "
    f"TBLPROPERTIES ('table_type'='ICEBERG', 'format'='parquet')"
)
# Load bronze from raw seed
run_athena(
    f"INSERT INTO {DATABASE_NAME}.{BRONZE_TABLE} "
    f"SELECT event_id, date(event_date) AS event_date, customer_id, revenue_cents "
    f"FROM (SELECT * FROM (VALUES (1)) t(x) "
    f"     CROSS JOIN (SELECT * FROM external_function_placeholder)) "
    f"WHERE 1=0"  # we instead bulk-load via DataLake INSERT below
)
# Simpler: load bronze using a Spark-free approach: create the table as
# external Glue table over the raw S3, then INSERT into Iceberg.
run_athena(f"DROP TABLE IF EXISTS {DATABASE_NAME}.raw_events_external")
glue.create_table(DatabaseName=DATABASE_NAME, TableInput={
    "Name": "raw_events_external",
    "TableType": "EXTERNAL_TABLE",
    "Parameters": {"classification": "json"},
    "StorageDescriptor": {
        "Columns": [
            {"Name": "event_id", "Type": "string"},
            {"Name": "event_date", "Type": "string"},
            {"Name": "customer_id", "Type": "string"},
            {"Name": "revenue_cents", "Type": "bigint"},
        ],
        "Location": f"s3://{BUCKET_NAME}/raw/",
        "InputFormat": "org.apache.hadoop.mapred.TextInputFormat",
        "OutputFormat": "org.apache.hadoop.hive.ql.io.HiveIgnoreKeyTextOutputFormat",
        "SerdeInfo": {"SerializationLibrary": "org.openx.data.jsonserde.JsonSerDe"},
    },
})

# Truth table: the correct daily aggregate (independent computation)
run_athena(
    f"CREATE TABLE {DATABASE_NAME}.{TRUTH_TABLE} ("
    f"  event_date date, revenue_cents bigint, order_count bigint"
    f") LOCATION 's3://{BUCKET_NAME}/{TRUTH_TABLE}/' "
    f"TBLPROPERTIES ('table_type'='ICEBERG', 'format'='parquet')"
)
run_athena(
    f"INSERT INTO {DATABASE_NAME}.{TRUTH_TABLE} "
    f"SELECT date(event_date) AS event_date, sum(revenue_cents) AS revenue_cents, count(*) AS order_count "
    f"FROM {DATABASE_NAME}.raw_events_external GROUP BY date(event_date)"
)

# Aggregate table: same shape, but the last 30 days are EMPTY (the bug).
run_athena(
    f"CREATE TABLE {DATABASE_NAME}.{AGG_TABLE} ("
    f"  event_date date, revenue_cents bigint, order_count bigint"
    f") LOCATION 's3://{BUCKET_NAME}/{AGG_TABLE}/' "
    f"TBLPROPERTIES ('table_type'='ICEBERG', 'format'='parquet')"
)
# Seed agg with only the 30 days OUTSIDE the backfill window (days 30-59).
run_athena(
    f"INSERT INTO {DATABASE_NAME}.{AGG_TABLE} "
    f"SELECT event_date, revenue_cents, order_count "
    f"FROM {DATABASE_NAME}.{TRUTH_TABLE} "
    f"WHERE event_date < date '{BACKFILL_START.isoformat()}'"
)
print(f"  truth + agg seeded; agg is missing the past 30 days ({BACKFILL_START} through {BACKFILL_END})")
print("Bronze ingest complete via Glue catalog. Bronze events seeded.")

# Load the bronze Iceberg table from the raw external table.
run_athena(
    f"INSERT INTO {DATABASE_NAME}.{BRONZE_TABLE} "
    f"SELECT event_id, date(event_date) AS event_date, customer_id, revenue_cents "
    f"FROM {DATABASE_NAME}.raw_events_external"
)


## Task 1: Build the incremental Glue job with bookmarks ENABLED and run it once

You stand up the Glue IAM role, the incremental ETL script, the Glue job with `--job-bookmark-option = job-bookmark-enable`, and run it once. The script reads bronze via `create_dynamic_frame.from_catalog` with a stable `transformation_ctx` so the bookmark persists across runs. The bookmark advances to today's watermark.

Capture the bookmark's RunId here; Task 4 will verify it is unchanged after the backfill.


In [ ]:
# NBVAL_SKIP
# Task 1: IAM role + incremental Glue job + first run.

trust_policy = {"Version": "2012-10-17", "Statement": [{"Effect": "Allow", "Principal": {"Service": "glue.amazonaws.com"}, "Action": "sts:AssumeRole"}]}
bucket_arn = f"arn:aws:s3:::{BUCKET_NAME}"
inline_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {"Effect": "Allow", "Action": ["s3:*"], "Resource": [bucket_arn, f"{bucket_arn}/*"]},
        {"Effect": "Allow", "Action": ["athena:*", "glue:*", "logs:*"], "Resource": "*"},
    ],
}

# YOUR CODE: create the IAM role + attach AWSGlueServiceRole + put_role_policy

iam.attach_role_policy(RoleName=ROLE_NAME, PolicyArn="arn:aws:iam::aws:policy/service-role/AWSGlueServiceRole")
# (The line above will fail if you haven't created the role yet; create it first via YOUR CODE.)

print("Your IAM role is stuck in traffic, give it 15 seconds...")
time.sleep(15)
role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{ROLE_NAME}"

# Incremental ETL script.
INCREMENTAL_SCRIPT = '''
import sys
from awsglue.context import GlueContext
from awsglue.job import Job
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from pyspark.sql.functions import col, count, sum as spark_sum, to_date

args = getResolvedOptions(sys.argv, ["JOB_NAME", "DATABASE_NAME", "BRONZE_TABLE", "AGG_TABLE"])
sc = SparkContext.getOrCreate()
glue_ctx = GlueContext(sc)
spark = glue_ctx.spark_session
job = Job(glue_ctx)
job.init(args["JOB_NAME"], args)

database = args["DATABASE_NAME"]
bronze_table = args["BRONZE_TABLE"]
agg_table = args["AGG_TABLE"]

dyf = glue_ctx.create_dynamic_frame.from_catalog(
    database=database, table_name=bronze_table, transformation_ctx="incremental_bronze_v1",
)
df = dyf.toDF()
rows_read = df.count()
print("LABRUN_BOOKMARK_OBSERVATION rows_read=" + str(rows_read))

if rows_read == 0:
    print("LABRUN_BOOKMARK_OBSERVATION no_new_partitions_since_last_bookmark")
    job.commit()
    sys.exit(0)

agg = (
    df.withColumn("event_date", to_date(col("event_date")))
      .groupBy("event_date")
      .agg(spark_sum("revenue_cents").alias("revenue_cents"), count("*").alias("order_count"))
)
agg.createOrReplaceTempView("agg_delta")

merge_sql = (
    "MERGE INTO glue_catalog." + database + "." + agg_table + " t "
    "USING agg_delta s ON t.event_date = s.event_date "
    "WHEN MATCHED THEN UPDATE SET revenue_cents = s.revenue_cents, order_count = s.order_count "
    "WHEN NOT MATCHED THEN INSERT (event_date, revenue_cents, order_count) "
    "                       VALUES (s.event_date, s.revenue_cents, s.order_count)"
)
spark.sql(merge_sql)
job.commit()
'''.lstrip()

incremental_script_uri = f"s3://{BUCKET_NAME}/scripts/incremental.py"
s3.put_object(Bucket=BUCKET_NAME, Key="scripts/incremental.py", Body=INCREMENTAL_SCRIPT.encode("utf-8"))

iceberg_conf = (
    "spark.sql.catalog.glue_catalog=org.apache.iceberg.spark.SparkCatalog "
    f"--conf spark.sql.catalog.glue_catalog.warehouse=s3://{BUCKET_NAME}/warehouse/ "
    "--conf spark.sql.catalog.glue_catalog.catalog-impl=org.apache.iceberg.aws.glue.GlueCatalog "
    "--conf spark.sql.catalog.glue_catalog.io-impl=org.apache.iceberg.aws.s3.S3FileIO "
    "--conf spark.sql.extensions=org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
)
incremental_default_args = {
    "--DATABASE_NAME": DATABASE_NAME,
    "--BRONZE_TABLE": BRONZE_TABLE,
    "--AGG_TABLE": AGG_TABLE,
    "--job-bookmark-option": "job-bookmark-enable",
    "--datalake-formats": "iceberg",
    "--conf": iceberg_conf,
    "--TempDir": f"s3://{BUCKET_NAME}/temp/",
    "--enable-metrics": "true",
}

# YOUR CODE: create the incremental Glue job (bookmarks ENABLED).
# YOUR CODE: start a run; poll until JobRunState in ("SUCCEEDED", "FAILED", ...).
# Capture the bookmark's RunId in INITIAL_BOOKMARK_RUN_ID.

# Create EventBridge daily schedule (disabled by default; declared for cleanup).
try:
    events.put_rule(Name=DAILY_RULE, ScheduleExpression="cron(0 9 * * ? *)", State="DISABLED", Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}])
except ClientError:
    pass


In [ ]:
# NBVAL_SKIP
# Checkpoint 1: bookmark advanced.

def validator_1(session):
    try:
        bm = glue.get_job_bookmark(JobName=INCREMENTAL_JOB)
    except glue.exceptions.EntityNotFoundException:
        return CheckpointResult("fail", error_reason=f"Glue job {INCREMENTAL_JOB} not found.")
    state = bm.get("JobBookmark") or {}
    run_id = state.get("RunId")
    if not run_id:
        return CheckpointResult("fail", error_reason="Bookmark has no RunId; did the incremental job complete successfully?")
    globals()["INITIAL_BOOKMARK_RUN_ID"] = run_id
    return CheckpointResult("pass")


check(1, validator_1)


<details><summary>Hint 1 (nudge)</summary>

Bookmarks only persist for reads via `create_dynamic_frame.from_catalog` with a stable `transformation_ctx`. The Glue job config sets `--job-bookmark-option = job-bookmark-enable`.

</details>

<details><summary>Hint 2 (direction)</summary>

Create the role (with the AWSGlueServiceRole managed policy + the inline policy). Wait 15s for IAM propagation. Create the job with the script above + `incremental_default_args`. Start a run. Poll `get_job_run` for JobRunState.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
iam.create_role(RoleName=ROLE_NAME, AssumeRolePolicyDocument=json.dumps(trust_policy), Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}])
iam.put_role_policy(RoleName=ROLE_NAME, PolicyName=INLINE_POLICY_NAME, PolicyDocument=json.dumps(inline_policy))

glue.create_job(
    Name=INCREMENTAL_JOB, Role=role_arn,
    Command={"Name": "glueetl", "ScriptLocation": incremental_script_uri, "PythonVersion": "3"},
    DefaultArguments=incremental_default_args,
    GlueVersion="4.0", WorkerType="G.1X", NumberOfWorkers=2, Timeout=15,
    Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
)

rid = glue.start_job_run(JobName=INCREMENTAL_JOB)["JobRunId"]
deadline = time.time() + 900
while time.time() < deadline:
    state = glue.get_job_run(JobName=INCREMENTAL_JOB, RunId=rid)["JobRun"]["JobRunState"]
    if state in ("SUCCEEDED", "FAILED", "STOPPED", "ERROR", "TIMEOUT"):
        break
    time.sleep(15)
print(state)
```

</details>


**Wallet check.** First Glue ETL run at the 10-minute minimum: ~$0.15. Athena scans during seed: pennies. Session total so far: ~16 cents.

## Task 2: Configure the backfill Glue job with bookmarks DISABLED and a 30-day window

The backfill job must NOT share bookmark state with the live job. Set `--job-bookmark-option = job-bookmark-disable`. Pass the window dates as Glue job parameters.

No execution in this task. The configuration is the deliverable.


In [ ]:
# NBVAL_SKIP
# Task 2: backfill job config.

BACKFILL_SCRIPT = '''
import sys
from awsglue.context import GlueContext
from awsglue.job import Job
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from pyspark.sql.functions import col, count, sum as spark_sum, to_date

args = getResolvedOptions(sys.argv, [
    "JOB_NAME", "DATABASE_NAME", "BRONZE_TABLE", "AGG_TABLE",
    "backfill_start_date", "backfill_end_date",
])
sc = SparkContext.getOrCreate()
glue_ctx = GlueContext(sc)
spark = glue_ctx.spark_session
job = Job(glue_ctx)
job.init(args["JOB_NAME"], args)

database = args["DATABASE_NAME"]
bronze_table = args["BRONZE_TABLE"]
agg_table = args["AGG_TABLE"]
start_date = args["backfill_start_date"]
end_date = args["backfill_end_date"]

print("LABRUN_BACKFILL_WINDOW start=" + start_date + " end=" + end_date)

# IMPORTANT: no transformation_ctx; backfill must not advance any bookmark.
df = (
    spark.read.table("glue_catalog." + database + "." + bronze_table)
         .withColumn("event_date", to_date(col("event_date")))
)
df = df.filter((col("event_date") >= start_date) & (col("event_date") <= end_date))
rows_read = df.count()
print("LABRUN_BACKFILL_OBSERVATION rows_read=" + str(rows_read))

agg = (
    df.groupBy("event_date")
      .agg(spark_sum("revenue_cents").alias("revenue_cents"), count("*").alias("order_count"))
)
agg.createOrReplaceTempView("agg_delta")

merge_sql = (
    "MERGE INTO glue_catalog." + database + "." + agg_table + " t "
    "USING agg_delta s ON t.event_date = s.event_date "
    "WHEN MATCHED THEN UPDATE SET revenue_cents = s.revenue_cents, order_count = s.order_count "
    "WHEN NOT MATCHED THEN INSERT (event_date, revenue_cents, order_count) "
    "                       VALUES (s.event_date, s.revenue_cents, s.order_count)"
)
spark.sql(merge_sql)
job.commit()
'''.lstrip()

backfill_script_uri = f"s3://{BUCKET_NAME}/scripts/backfill.py"
# YOUR CODE: upload the backfill script to S3.

backfill_default_args = {
    "--DATABASE_NAME": DATABASE_NAME,
    "--BRONZE_TABLE": BRONZE_TABLE,
    "--AGG_TABLE": AGG_TABLE,
    "--backfill_start_date": BACKFILL_START.isoformat(),
    "--backfill_end_date": BACKFILL_END.isoformat(),
    "--job-bookmark-option": "job-bookmark-disable",
    "--datalake-formats": "iceberg",
    "--conf": iceberg_conf,
    "--TempDir": f"s3://{BUCKET_NAME}/temp/",
}

# YOUR CODE: create the backfill Glue job with bookmarks DISABLED. No
# run yet; Task 3 starts it via Step Functions.


In [ ]:
# NBVAL_SKIP
# Checkpoint 2: backfill job config has bookmarks disabled + window args.

def validator_2(session):
    try:
        job = glue.get_job(JobName=BACKFILL_JOB)["Job"]
    except glue.exceptions.EntityNotFoundException:
        return CheckpointResult("fail", error_reason=f"Backfill Glue job {BACKFILL_JOB} not found.")
    args = job.get("DefaultArguments", {})
    if args.get("--job-bookmark-option") != "job-bookmark-disable":
        return CheckpointResult("fail", error_reason=f"--job-bookmark-option is {args.get('--job-bookmark-option')!r}; expected 'job-bookmark-disable'.")
    if "--backfill_start_date" not in args or "--backfill_end_date" not in args:
        return CheckpointResult("fail", error_reason="Backfill window dates missing from DefaultArguments. Pass --backfill_start_date and --backfill_end_date.")
    return CheckpointResult("pass")


check(2, validator_2)


<details><summary>Hint 1 (nudge)</summary>

Backfills must not share bookmark state with the live job.

</details>

<details><summary>Hint 2 (direction)</summary>

`--job-bookmark-option = job-bookmark-disable` on the backfill job's DefaultArguments. Window dates are also DefaultArguments (`--backfill_start_date`, `--backfill_end_date`).

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
s3.put_object(Bucket=BUCKET_NAME, Key="scripts/backfill.py", Body=BACKFILL_SCRIPT.encode("utf-8"))

glue.create_job(
    Name=BACKFILL_JOB, Role=role_arn,
    Command={"Name": "glueetl", "ScriptLocation": backfill_script_uri, "PythonVersion": "3"},
    DefaultArguments=backfill_default_args,
    GlueVersion="4.0", WorkerType="G.1X", NumberOfWorkers=2, Timeout=15,
    Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
)
```

</details>


**Wallet check.** Configuring the backfill job is a control-plane call: free. Uploading the script is free. Session total so far: ~16 cents.

## Task 3: Run the backfill via Step Functions; scan stays under 30 GB

Build a one-state Step Functions state machine that wraps `glue:startJobRun.sync` on the backfill job. Start the execution. Wait for SUCCEEDED. After completion, `daily_revenue_agg` should have rows for all 30 days of the window.


In [ ]:
# NBVAL_SKIP
# Task 3: SFN state machine + execution.

sfn_trust = {"Version": "2012-10-17", "Statement": [{"Effect": "Allow", "Principal": {"Service": "states.amazonaws.com"}, "Action": "sts:AssumeRole"}]}
sfn_inline = {
    "Version": "2012-10-17",
    "Statement": [
        {"Effect": "Allow", "Action": ["glue:StartJobRun", "glue:GetJobRun", "glue:GetJobRuns", "glue:BatchStopJobRun"], "Resource": "*"},
        {"Effect": "Allow", "Action": ["iam:PassRole"], "Resource": role_arn},
        {"Effect": "Allow", "Action": ["events:PutTargets", "events:PutRule", "events:DescribeRule"], "Resource": "*"},
    ],
}

try:
    iam.create_role(RoleName=SFN_ROLE, AssumeRolePolicyDocument=json.dumps(sfn_trust), Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}])
    iam.put_role_policy(RoleName=SFN_ROLE, PolicyName="labrun-sfn-inline", PolicyDocument=json.dumps(sfn_inline))
except iam.exceptions.EntityAlreadyExistsException:
    pass
print("SFN IAM propagation, hold 15 seconds...")
time.sleep(15)
SFN_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{SFN_ROLE}"

asl = {
    "Comment": "Run the backfill Glue job",
    "StartAt": "RunBackfill",
    "States": {
        "RunBackfill": {
            "Type": "Task",
            "Resource": "arn:aws:states:::glue:startJobRun.sync",
            "Parameters": {"JobName": BACKFILL_JOB},
            "End": True,
        }
    },
}

# YOUR CODE: create the state machine with sfn.create_state_machine().
# Capture the stateMachineArn into SM_ARN; also update CLEANUP_MANIFEST's
# sfn_state_machine entry.

# YOUR CODE: start the execution; poll sfn.describe_execution until
# status in ("SUCCEEDED", "FAILED", ...).


In [ ]:
# NBVAL_SKIP
# Checkpoint 3: backfill SUCCEEDED + agg has 30 rows in window + scan <30 GB.

def validator_3(session):
    # Step Functions execution succeeded?
    if not SM_ARN:
        return CheckpointResult("fail", error_reason="SM_ARN not captured; did you create the state machine and start the execution?")
    try:
        executions = sfn.list_executions(stateMachineArn=SM_ARN, maxResults=5)["executions"]
    except ClientError as exc:
        return CheckpointResult("error", error_reason=f"SFN list_executions failed: {exc}")
    if not executions:
        return CheckpointResult("fail", error_reason="No SFN executions found.")
    latest = executions[0]
    if latest["status"] != "SUCCEEDED":
        return CheckpointResult("fail", error_reason=f"Latest SFN execution status is {latest['status']}; expected SUCCEEDED.")

    # Agg has 30 days?
    try:
        qid = athena.start_query_execution(
            QueryString=f"SELECT count(*) FROM {DATABASE_NAME}.{AGG_TABLE} WHERE event_date BETWEEN date '{BACKFILL_START.isoformat()}' AND date '{BACKFILL_END.isoformat()}'",
            WorkGroup=ATHENA_WG, QueryExecutionContext={"Database": DATABASE_NAME},
        )["QueryExecutionId"]
        deadline = time.time() + 60
        while time.time() < deadline:
            s = athena.get_query_execution(QueryExecutionId=qid)["QueryExecution"]["Status"]["State"]
            if s == "SUCCEEDED":
                break
            if s in ("FAILED", "CANCELLED"):
                return CheckpointResult("error", error_reason=f"Athena query {s}")
            time.sleep(2)
        in_window = int(athena.get_query_results(QueryExecutionId=qid)["ResultSet"]["Rows"][1]["Data"][0]["VarCharValue"])
        if in_window != 30:
            return CheckpointResult("fail", error_reason=f"daily_revenue_agg has {in_window} rows in the backfill window; expected 30.")
    except Exception as exc:
        return CheckpointResult("error", error_reason=f"Validator failed: {exc}")
    return CheckpointResult("pass")


check(3, validator_3)


<details><summary>Hint 1 (nudge)</summary>

Step Functions has a built-in integration for Glue: `arn:aws:states:::glue:startJobRun.sync`. The state machine waits for the Glue job to complete.

</details>

<details><summary>Hint 2 (direction)</summary>

Create the SFN role with `glue:StartJobRun` + `iam:PassRole` on the Glue role. Define a one-state ASL. Start the execution and poll.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
sm = sfn.create_state_machine(name=SFN_NAME, definition=json.dumps(asl), roleArn=SFN_ROLE_ARN, tags=[{"key": LAB_TAG_KEY, "value": LAB_TAG_VALUE}])
SM_ARN = sm["stateMachineArn"]
for r in CLEANUP_MANIFEST:
    if r.type == "sfn_state_machine":
        r.id = SM_ARN
        break

exec_arn = sfn.start_execution(stateMachineArn=SM_ARN, input="{}")["executionArn"]
deadline = time.time() + 900
while time.time() < deadline:
    d = sfn.describe_execution(executionArn=exec_arn)
    if d["status"] in ("SUCCEEDED", "FAILED", "TIMED_OUT", "ABORTED"):
        print(d["status"])
        break
    time.sleep(15)
```

</details>


**Wallet check.** Second Glue ETL run at the 10-minute minimum: another ~$0.15. Step Functions: free at lab volume. Session total so far: ~32 cents.

## Task 4: Verify the live bookmark is unchanged after the backfill

The constraint that earns you the interview talking point: the backfill must not poke the live job's bookmark state. Read the bookmark again, compare to `INITIAL_BOOKMARK_RUN_ID` from Task 1.


In [ ]:
# NBVAL_SKIP
# Task 4: verify bookmark unchanged.

# YOUR CODE: read glue.get_job_bookmark(JobName=INCREMENTAL_JOB) and
# print the current RunId. Compare it to INITIAL_BOOKMARK_RUN_ID (captured
# in Task 1's checkpoint).


In [ ]:
# NBVAL_SKIP
# Checkpoint 4: bookmark RunId equals the value captured in Task 1.

def validator_4(session):
    if not INITIAL_BOOKMARK_RUN_ID:
        return CheckpointResult("fail", error_reason="INITIAL_BOOKMARK_RUN_ID was not captured in Task 1. Re-run Checkpoint 1 first.")
    try:
        bm = glue.get_job_bookmark(JobName=INCREMENTAL_JOB)
    except glue.exceptions.EntityNotFoundException:
        return CheckpointResult("fail", error_reason=f"Incremental Glue job {INCREMENTAL_JOB} not found.")
    current = (bm.get("JobBookmark") or {}).get("RunId")
    if current != INITIAL_BOOKMARK_RUN_ID:
        return CheckpointResult("fail", error_reason=f"Bookmark RunId changed from {INITIAL_BOOKMARK_RUN_ID!r} to {current!r}. The backfill advanced the live bookmark.")
    return CheckpointResult("pass")


check(4, validator_4)


<details><summary>Hint 1 (nudge)</summary>

Read the live job's bookmark; compare to the captured value.

</details>

<details><summary>Hint 2 (direction)</summary>

`glue.get_job_bookmark(JobName=INCREMENTAL_JOB)["JobBookmark"]["RunId"]` should equal `INITIAL_BOOKMARK_RUN_ID`. If they differ, the backfill job had bookmarks enabled and advanced state.

</details>

<details><summary>Hint 3 (spoiler)</summary>

No code change needed; the validator does the comparison. Just run the checkpoint cell.

</details>


**Wallet check.** Reading a bookmark via get_job_bookmark is a control-plane call: free. Session total: ~32 cents.

## Task 5: Row-level parity diff against `daily_revenue_truth`

The deliverable. A FULL OUTER JOIN between `daily_revenue_agg` and `daily_revenue_truth` filtered to the backfill window must materialize zero mismatching rows.


In [ ]:
# NBVAL_SKIP
# Task 5: parity diff.

# YOUR CODE: run the parity diff query against agg vs truth on the
# backfill window. Confirm zero mismatching rows.


In [ ]:
# NBVAL_SKIP
# Checkpoint 5: parity diff returns zero rows.

def validator_5(session):
    diff_query = (
        f"SELECT t.event_date, a.revenue_cents AS agg_revenue, t.revenue_cents AS truth_revenue, "
        f"       a.order_count AS agg_orders, t.order_count AS truth_orders "
        f"FROM {DATABASE_NAME}.{TRUTH_TABLE} t FULL OUTER JOIN {DATABASE_NAME}.{AGG_TABLE} a "
        f"  ON t.event_date = a.event_date "
        f"WHERE t.event_date BETWEEN date '{BACKFILL_START.isoformat()}' AND date '{BACKFILL_END.isoformat()}' "
        f"  AND (a.event_date IS NULL OR t.event_date IS NULL "
        f"       OR a.revenue_cents <> t.revenue_cents OR a.order_count <> t.order_count)"
    )
    try:
        qid = athena.start_query_execution(QueryString=f"SELECT count(*) FROM ({diff_query})", WorkGroup=ATHENA_WG, QueryExecutionContext={"Database": DATABASE_NAME})["QueryExecutionId"]
        deadline = time.time() + 60
        while time.time() < deadline:
            s = athena.get_query_execution(QueryExecutionId=qid)["QueryExecution"]["Status"]["State"]
            if s == "SUCCEEDED":
                break
            if s in ("FAILED", "CANCELLED"):
                return CheckpointResult("error", error_reason=f"Athena query {s}")
            time.sleep(2)
        count = int(athena.get_query_results(QueryExecutionId=qid)["ResultSet"]["Rows"][1]["Data"][0]["VarCharValue"])
    except Exception as exc:
        return CheckpointResult("error", error_reason=f"Validator failed: {exc}")
    if count != 0:
        return CheckpointResult("fail", error_reason=f"Parity diff returns {count} mismatching rows; expected 0.")
    return CheckpointResult("pass")


check(5, validator_5)


<details><summary>Hint 1 (nudge)</summary>

FULL OUTER JOIN with NULL detection on either side catches missing rows in either table.

</details>

<details><summary>Hint 2 (direction)</summary>

Round revenue_cents stays BIGINT all the way through; compare exactly. Tolerance is a smell on monetary aggregates.

</details>

<details><summary>Hint 3 (spoiler)</summary>

The validator runs the diff for you. If you want to inspect what mismatches look like, run:

```sql
SELECT t.event_date, a.revenue_cents AS agg_revenue, t.revenue_cents AS truth_revenue
FROM daily_revenue_truth t FULL OUTER JOIN daily_revenue_agg a
  ON t.event_date = a.event_date
WHERE t.event_date BETWEEN date '...' AND date '...'
  AND (a.event_date IS NULL OR t.event_date IS NULL
       OR a.revenue_cents <> t.revenue_cents OR a.order_count <> t.order_count)
```

</details>


**Wallet check.** One small Athena scan: under a cent. Session total so far: ~32 cents.

In [ ]:
# NBVAL_SKIP
from labrun_checks.adapters.aws import AwsCleanupAdapter


class _LabAdapter(AwsCleanupAdapter):
    def delete_resource(self, credentials, resource):
        if resource.type == "iceberg_table":
            db, table = resource.id.split(".", 1)
            client = boto3.client("glue", region_name=resource.region, **{k: v for k, v in credentials.items() if v})
            try:
                client.delete_table(DatabaseName=db, Name=table)
            except client.exceptions.EntityNotFoundException:
                pass
            return
        if resource.type == "athena_workgroup":
            client = boto3.client("athena", region_name=resource.region, **{k: v for k, v in credentials.items() if v})
            try:
                client.delete_work_group(WorkGroup=resource.id, RecursiveDeleteOption=True)
            except Exception:
                pass
            return
        if resource.type == "sfn_state_machine":
            client = boto3.client("stepfunctions", region_name=resource.region, **{k: v for k, v in credentials.items() if v})
            try:
                client.delete_state_machine(stateMachineArn=resource.id)
            except ClientError:
                pass
            return
        if resource.type == "eventbridge_rule":
            client = boto3.client("events", region_name=resource.region, **{k: v for k, v in credentials.items() if v})
            try:
                for t in client.list_targets_by_rule(Rule=resource.id).get("Targets", []):
                    client.remove_targets(Rule=resource.id, Ids=[t["Id"]])
                client.delete_rule(Name=resource.id)
            except ClientError:
                pass
            return
        return super().delete_resource(credentials, resource)

    def describe_resource(self, credentials, resource):
        if resource.type in ("iceberg_table", "athena_workgroup", "sfn_state_machine", "eventbridge_rule"):
            return False
        return super().describe_resource(credentials, resource)


# Drop the raw_events_external external table (not in manifest).
try:
    glue.delete_table(DatabaseName=DATABASE_NAME, Name="raw_events_external")
except glue.exceptions.EntityNotFoundException:
    pass

result = run_cleanup(CLEANUP_MANIFEST, adapter=_LabAdapter())

print()
print("Cleanup complete.")
critical_count = sum(1 for r in CLEANUP_MANIFEST if r.critical)
standard_count = len(CLEANUP_MANIFEST) - critical_count
print(f"Critical resources destroyed: {critical_count}")
print(f"Standard resources destroyed: {standard_count}")
failed = len(result.warnings) + len(result.orphans)
print(f"Resources that failed to delete: {failed}")
if failed > 0:
    print("If failed > 0, your AWS account may still incur charges. Resolve before closing this notebook.")
    for w in result.warnings:
        print(f"  WARNING: {w}")
    for o in result.orphans:
        print(f"  ORPHAN: {o}")

cleanup(status=result.status)
if result.status == "dirty":
    sys.exit(1)


**Session total: $0.80 to $1.50.** Two Glue ETL runs are the only meaningful spend. Athena scans for parity + validators: pennies. If you debugged a script bug and reran the backfill, add ~$0.15 per rerun.

## These are not graded. They are for you.

**1. The failure mode of advancing the live bookmark by mistake.** Walk through what happens to tomorrow's incremental run if your backfill accidentally enabled bookmarks. What appears in the next morning's run output?

**2. Idempotency on retry.** The lab uses MERGE INTO so a second backfill run produces the same final state as a single run. Walk through an alternative pattern (DELETE-then-INSERT inside a transaction). When would you pick each?

## Exam-Style Practice

**Question 1.** Your backfill job needs to scan only the 30-day window. Which Glue config is the lever?

A) Job bookmarks; pass the window via the bookmark state.

B) Job parameters plus a DataFrame filter on the read.

C) A Glue crawler with a partition filter.

D) Glue Studio's visual partition-pruning transform.

<details><summary>Show answer</summary>

**B**.

- A) Wrong. Bookmarks track which partitions you have already consumed; they are not a way to scope a one-off backfill to a window.
- B) Right. Pass the window as Glue job DefaultArguments (`--backfill_start_date`, `--backfill_end_date`); read via `spark.read.table(...)` then `.filter(col("event_date").between(start, end))`. The filter pushes down to the Iceberg scan.
- C) Wrong. A crawler discovers tables; it does not scope a job's read.
- D) Wrong. Glue Studio is a visual UI; the partition-pruning transform doesn't exist as a first-class operator with a "window" parameter.

</details>

**Question 2.** Your backfill writes daily_revenue_agg via plain INSERT. The job retries after a network failure on commit. What is the most likely outcome?

A) The retry produces an idempotent rebuild; no data is duplicated.

B) The retry produces duplicate rows in the agg table for the dates the first attempt already wrote.

C) The retry is rejected by Iceberg's snapshot isolation.

D) The retry's INSERT is no-op'd by AWS's at-most-once delivery on Glue commits.

<details><summary>Show answer</summary>

**B**.

- A) Wrong. Plain INSERT is not idempotent under retry. The first attempt's writes are committed; the second attempt appends again.
- B) Right. The retry produces duplicates. The fix is MERGE INTO (or DELETE-then-INSERT in a single transaction).
- C) Wrong. Iceberg's snapshot isolation prevents readers from seeing partial writes; it does NOT prevent the writer from inserting duplicate rows on a retry.
- D) Wrong. AWS does not guarantee at-most-once delivery on Glue. Retries are exactly the failure mode this question tests.

</details>
